# 🤖 Fine-Tuning LLM for Sanaullah AI Assistant (LoRA / QLoRA)
This notebook orchestrates Parameter-Efficient Fine-Tuning (PEFT/LoRA) on open-source LLMs (Qwen / LLaMA-3 / Mistral) using Hugging Face Transformers and PyTorch.

In [ ]:
!pip install -q torch transformers datasets peft accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto'
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Load paired dataset
import json
from datasets import Dataset

with open('../data/train_dataset.json', 'r') as f:
    raw_data = json.load(f)

formatted = [{'text': f"<|im_start|>user\n{x['instruction']}<|im_end|>\n<|im_start|>assistant\n{x['output']}<|im_end|>"} for x in raw_data]
dataset = Dataset.from_dict({'text': [x['text'] for x in formatted]})
tokenized_ds = dataset.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=512), batched=True)
print('Dataset ready:', len(tokenized_ds))

In [ ]:
training_args = TrainingArguments(
    output_dir='../models/lora_adapter',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    fp16=torch.cuda.is_available(),
    save_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds
)

trainer.train()
model.save_pretrained('../models/lora_adapter')
tokenizer.save_pretrained('../models/tokenizer')